# LeetCode 278 — First Bad Version bằng Python

**Link đề:** https://leetcode.com/problems/first-bad-version/
**Độ khó:** Easy · **Chủ đề:** Binary Search, Interactive

```python
# The isBadVersion API is already defined for you.
# def isBadVersion(version: int) -> bool:

class Solution:
    def firstBadVersion(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài **interactive**: không có mảng, chỉ có một **API** để hỏi — và phải **hỏi ít nhất có thể**.
2. Tự giả lập API `isBadVersion` có **đếm số lần gọi** để đo hiệu quả.
3. Nhận ra dãy kết quả có dạng **đơn điệu** `F F F ... F T T ... T` → tìm "điểm chuyển" bằng **binary search**.
4. Viết binary search chuẩn `lo < hi` (tìm vị trí đầu tiên thỏa điều kiện), trace từng bước và chứng minh bằng invariant.
5. So sánh với khuôn `lo <= hi` + biến `ans`, và `bisect` có `key`; biết các lỗi off-by-one kinh điển.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Bạn có `n` phiên bản `1, 2, ..., n`. Từ một phiên bản **lỗi** nào đó trở đi, **mọi** phiên bản sau đều lỗi (vì phát triển dựa trên bản trước). Tìm **phiên bản lỗi đầu tiên**.

Bạn được cho API `isBadVersion(version) -> bool`. Hãy **gọi API ít lần nhất**.

**Ví dụ 1:** `n = 5, bad = 4` → `4`

```text
isBadVersion(3) -> False
isBadVersion(5) -> True
isBadVersion(4) -> True
=> 4 là phiên bản lỗi đầu tiên
```

**Ví dụ 2:** `n = 1, bad = 1` → `1`

**Constraints:** `1 <= bad <= n <= 2^31 - 1` → luôn có ít nhất một bản lỗi, và `n` có thể **rất lớn** (hơn 2 tỷ).

In [1]:
n, bad = 5, 4
print("Trạng thái các phiên bản:", ["T" if v >= bad else "F" for v in range(1, n + 1)])

Trạng thái các phiên bản: ['F', 'F', 'F', 'T', 'T']


## 2. Nền tảng

### 2.1. Giả lập API trong notebook

Trên LeetCode, `isBadVersion` đã có sẵn. Ở đây ta tự tạo: hàm `set_bad(bad)` gán một hàm **toàn cục** `isBadVersion` mới, kèm **bộ đếm** số lần gọi `CALLS`.

- `global` cho phép một hàm gán lại biến cấp module.
- **Closure**: hàm `api` bên trong "nhớ" giá trị `bad` của hàm ngoài ngay cả khi hàm ngoài đã chạy xong.

In [2]:
CALLS = 0

def set_bad(bad: int) -> None:
    global isBadVersion, CALLS
    CALLS = 0
    def api(version: int) -> bool:
        global CALLS
        CALLS += 1
        return version >= bad
    isBadVersion = api

set_bad(4)
print([isBadVersion(v) for v in range(1, 6)], "| số lần gọi:", CALLS)
set_bad(1)
print(isBadVersion(1), "| số lần gọi sau khi đặt lại:", CALLS)

[False, False, False, True, True] | số lần gọi: 5
True | số lần gọi sau khi đặt lại: 1


### 2.2. Chia nguyên `//` và điểm giữa

`mid = (lo + hi) // 2` lấy điểm giữa, **làm tròn xuống**. Ví dụ `(1 + 4) // 2 = 2`, `(3 + 4) // 2 = 3`.

Ở Java/C++, `lo + hi` có thể **tràn số** khi gần `2^31`; cách viết an toàn là `lo + (hi - lo) // 2`. Python không tràn, nhưng nên quen cách viết an toàn.

In [3]:
for lo, hi in [(1, 4), (3, 4), (1, 5), (2**31 - 2, 2**31 - 1)]:
    print(f"lo={lo}, hi={hi}: (lo+hi)//2 = {(lo + hi) // 2}, lo+(hi-lo)//2 = {lo + (hi - lo) // 2}")
print("lo + hi với số lớn =", (2**31 - 2) + (2**31 - 1), "> 2^31 - 1 =", 2**31 - 1, "(tràn ở int 32-bit)")

lo=1, hi=4: (lo+hi)//2 = 2, lo+(hi-lo)//2 = 2
lo=3, hi=4: (lo+hi)//2 = 3, lo+(hi-lo)//2 = 3
lo=1, hi=5: (lo+hi)//2 = 3, lo+(hi-lo)//2 = 3
lo=2147483646, hi=2147483647: (lo+hi)//2 = 2147483646, lo+(hi-lo)//2 = 2147483646
lo + hi với số lớn = 4294967293 > 2^31 - 1 = 2147483647 (tràn ở int 32-bit)


### 2.3. `math.log2` và `math.ceil` — đo số bước binary search

Mỗi bước binary search **chia đôi** khoảng tìm kiếm. Từ `n` còn 1 cần khoảng `ceil(log2(n))` bước.

In [4]:
import math
for n in [5, 1000, 10**6, 2**31 - 1]:
    print(f"n = {n:>10}: log2(n) ≈ {math.log2(n):6.2f} -> khoảng {math.ceil(math.log2(n))} lần gọi")

n =          5: log2(n) ≈   2.32 -> khoảng 3 lần gọi
n =       1000: log2(n) ≈   9.97 -> khoảng 10 lần gọi
n =    1000000: log2(n) ≈  19.93 -> khoảng 20 lần gọi
n = 2147483647: log2(n) ≈  31.00 -> khoảng 31 lần gọi


## 3. Cách 1 — Duyệt tuần tự (baseline)

Hỏi lần lượt `1, 2, 3, ...`; gặp `True` đầu tiên thì trả về. Đúng, nhưng có thể gọi API tới `n` lần (khi bản lỗi ở cuối) — với `n ≈ 2·10^9` là **Time Limit Exceeded**.

In [5]:
def first_bad_linear(n: int) -> int:
    for v in range(1, n + 1):
        if isBadVersion(v):
            return v
    return -1           # không xảy ra theo constraints

set_bad(4)
print(first_bad_linear(5), "| số lần gọi:", CALLS)
set_bad(99999)
print(first_bad_linear(100000), "| số lần gọi:", CALLS)

4 | số lần gọi: 4
99999 | số lần gọi: 99999


## 4. Nút thắt và insight — dãy đơn điệu `F...F T...T`

Kết quả `isBadVersion(1..n)` luôn có dạng:

```text
version: 1  2  3  4  5  6  7
result : F  F  F  T  T  T  T
                  ^ điểm chuyển = đáp án
```

Hỏi **một** điểm `mid` cho ta biết cả một nửa:
- `isBadVersion(mid) == True` → đáp án `<= mid` (có thể chính là `mid`) → bỏ nửa phải **sau** `mid`.
- `False` → đáp án `> mid` → bỏ nửa trái **kể cả** `mid`.

Mỗi lần hỏi loại được khoảng một nửa → `O(log n)` lần gọi. Với `n = 2^31 - 1`, chỉ ~31 lần.

Đây là dạng **"tìm vị trí đầu tiên thỏa điều kiện"** (lower bound) — khuôn binary search quan trọng nhất.

In [6]:
n, bad = 7, 4
lo, hi = 1, n
mid = (lo + hi) // 2
print(f"Hỏi mid = {mid}: {'T' if mid >= bad else 'F'} ->",
      "đáp án trong [lo, mid]" if mid >= bad else "đáp án trong [mid+1, hi]")

Hỏi mid = 4: T -> đáp án trong [lo, mid]


## 5. Cách 2 — Lời giải chính: binary search `lo < hi`, giảng từng đoạn

**Đoạn 1 — khoảng tìm kiếm:** `lo, hi = 1, n`. Ý nghĩa: **đáp án chắc chắn nằm trong `[lo, hi]`** (đóng hai đầu).

**Đoạn 2 — vòng lặp:** `while lo < hi:` — khi `lo == hi` khoảng chỉ còn một số → chính là đáp án.

**Đoạn 3 — hỏi điểm giữa:** `mid = lo + (hi - lo) // 2`.

**Đoạn 4 — thu hẹp:**
```python
if isBadVersion(mid):
    hi = mid          # mid có thể là đáp án -> GIỮ mid
else:
    lo = mid + 1      # mid tốt -> đáp án ở sau mid
```

**Đoạn 5:** `return lo` (lúc này `lo == hi`).

Vì sao vòng lặp chắc chắn dừng? `mid` làm tròn xuống nên `lo <= mid < hi`; cả hai nhánh đều làm khoảng **ngắn đi ít nhất 1**.

In [7]:
def first_bad_binary(n: int) -> int:
    lo, hi = 1, n
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if isBadVersion(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for n, bad in [(5, 4), (1, 1), (100000, 99999), (2**31 - 1, 2**31 - 1), (2**31 - 1, 1)]:
    set_bad(bad)
    ans = first_bad_binary(n)
    print(f"n = {n:>10}, bad = {bad:>10} -> {ans:>10} | số lần gọi: {CALLS}")

n =          5, bad =          4 ->          4 | số lần gọi: 2
n =          1, bad =          1 ->          1 | số lần gọi: 0
n =     100000, bad =      99999 ->      99999 | số lần gọi: 17
n = 2147483647, bad = 2147483647 -> 2147483647 | số lần gọi: 30
n = 2147483647, bad =          1 ->          1 | số lần gọi: 31


## 6. TRACE từng bước

`n = 10, bad = 7`:

| Bước | lo | hi | mid | isBad(mid) | hành động | khoảng mới |
|---:|---:|---:|---:|:---:|---|---|
| 1 | 1 | 10 | 5 | F | `lo = 6` | [6, 10] |
| 2 | 6 | 10 | 8 | T | `hi = 8` | [6, 8] |
| 3 | 6 | 8 | 7 | T | `hi = 7` | [6, 7] |
| 4 | 6 | 7 | 6 | F | `lo = 7` | [7, 7] |
| — | 7 | 7 | — | — | dừng | trả 7 |

4 lần gọi API (so với 7 lần nếu duyệt tuần tự).

In [8]:
def trace_first_bad(n: int, bad: int) -> int:
    set_bad(bad)
    lo, hi = 1, n
    step = 0
    print(f"n = {n}, bad (bí mật) = {bad}")
    while lo < hi:
        step += 1
        mid = lo + (hi - lo) // 2
        res = isBadVersion(mid)
        if res:
            action, hi = f"hi = {mid}", mid
        else:
            action, lo = f"lo = {mid + 1}", mid + 1
        print(f"  bước {step}: mid = {mid:>2} -> {'T' if res else 'F'} | {action:<8} | khoảng còn [{lo}, {hi}]")
    print(f"  lo == hi == {lo} -> đáp án {lo} | tổng số lần gọi API: {CALLS}")
    return lo

trace_first_bad(10, 7)
print()
trace_first_bad(5, 4)
print()
trace_first_bad(1, 1)

n = 10, bad (bí mật) = 7
  bước 1: mid =  5 -> F | lo = 6   | khoảng còn [6, 10]
  bước 2: mid =  8 -> T | hi = 8   | khoảng còn [6, 8]
  bước 3: mid =  7 -> T | hi = 7   | khoảng còn [6, 7]
  bước 4: mid =  6 -> F | lo = 7   | khoảng còn [7, 7]
  lo == hi == 7 -> đáp án 7 | tổng số lần gọi API: 4

n = 5, bad (bí mật) = 4
  bước 1: mid =  3 -> F | lo = 4   | khoảng còn [4, 5]
  bước 2: mid =  4 -> T | hi = 4   | khoảng còn [4, 4]
  lo == hi == 4 -> đáp án 4 | tổng số lần gọi API: 2

n = 1, bad (bí mật) = 1
  lo == hi == 1 -> đáp án 1 | tổng số lần gọi API: 0


1

Chú ý ca `n = 1`: vòng lặp **không chạy lần nào**, trả `1` mà **không gọi API** — hợp lệ vì đề đảm bảo có bản lỗi.

## 7. Cách 3 — Khuôn `lo <= hi` với biến `ans`

Khuôn khác cũng hay gặp: giữ `ans` = "bản lỗi nhỏ nhất đã thấy", luôn loại `mid` khỏi khoảng.

```python
lo, hi, ans = 1, n, n
while lo <= hi:
    mid = (lo + hi) // 2
    if isBadVersion(mid):
        ans = mid; hi = mid - 1     # ghi nhận rồi tìm tiếp bên trái
    else:
        lo = mid + 1
return ans
```

Ưu điểm: điều kiện cập nhật đối xứng, dễ nhớ. Nhược điểm: thêm một biến, và số lần gọi có thể nhiều hơn 1 so với khuôn `lo < hi`.

In [9]:
def first_bad_with_ans(n: int) -> int:
    lo, hi, ans = 1, n, n
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if isBadVersion(mid):
            ans = mid
            hi = mid - 1
        else:
            lo = mid + 1
    return ans

for n, bad in [(5, 4), (1, 1), (10, 7), (2**31 - 1, 123456789)]:
    set_bad(bad); a1 = first_bad_binary(n); c1 = CALLS
    set_bad(bad); a2 = first_bad_with_ans(n); c2 = CALLS
    print(f"n = {n}, bad = {bad}: lo<hi -> {a1} ({c1} lần) | lo<=hi -> {a2} ({c2} lần)")

n = 5, bad = 4: lo<hi -> 4 (2 lần) | lo<=hi -> 4 (2 lần)
n = 1, bad = 1: lo<hi -> 1 (0 lần) | lo<=hi -> 1 (1 lần)
n = 10, bad = 7: lo<hi -> 7 (4 lần) | lo<=hi -> 7 (4 lần)
n = 2147483647, bad = 123456789: lo<hi -> 123456789 (31 lần) | lo<=hi -> 123456789 (31 lần)


## 8. Cách 4 — Dùng thư viện: `bisect_left` với `key` (Python 3.10+)

`bisect.bisect_left(a, x, key=f)` tìm vị trí đầu tiên `i` mà `f(a[i]) >= x`. Với `a = range(1, n + 1)` (không tạo list thật, `range` hỗ trợ truy cập theo chỉ số `O(1)`), `x = True`, `key = isBadVersion` (vì `False < True` trong Python) → `i` là chỉ số của bản lỗi đầu tiên, đáp án là `i + 1`.

Rất gọn, nhưng **phỏng vấn thường muốn bạn tự viết** binary search. Và LeetCode phải chạy Python đủ mới để có tham số `key`.

In [10]:
import bisect
print("False < True:", False < True, "| range(1, 6)[3] =", range(1, 6)[3])

def first_bad_bisect(n: int) -> int:
    return bisect.bisect_left(range(1, n + 1), True, key=lambda v: isBadVersion(v)) + 1

for n, bad in [(5, 4), (1, 1), (2**31 - 1, 2**30)]:
    set_bad(bad)
    print(f"n = {n}, bad = {bad} -> {first_bad_bisect(n)} | số lần gọi: {CALLS}")

False < True: True | range(1, 6)[3] = 4
n = 5, bad = 4 -> 4 | số lần gọi: 3
n = 1, bad = 1 -> 1 | số lần gọi: 1
n = 2147483647, bad = 1073741824 -> 1073741824 | số lần gọi: 31


## 9. Code nộp LeetCode

In [11]:
# The isBadVersion API is already defined for you (ở notebook: do set_bad tạo ra).

class Solution:
    def firstBadVersion(self, n: int) -> int:
        lo, hi = 1, n
        while lo < hi:
            mid = lo + (hi - lo) // 2
            if isBadVersion(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

set_bad(4)
print(Solution().firstBadVersion(5), "| gọi", CALLS, "lần")
set_bad(1)
print(Solution().firstBadVersion(1), "| gọi", CALLS, "lần")

4 | gọi 2 lần
1 | gọi 0 lần


## 10. Vì sao đúng? (invariant) + độ phức tạp

**Invariant:** ở đầu mỗi vòng lặp, **phiên bản lỗi đầu tiên `B` nằm trong `[lo, hi]`**.
- Ban đầu `[1, n]` chứa `B` (constraints: `1 <= bad <= n`).
- Nếu `isBadVersion(mid)` đúng: `mid >= B`, gán `hi = mid` → `B` vẫn trong `[lo, mid]`.
- Nếu sai: `mid < B`, gán `lo = mid + 1` → `B` vẫn trong `[mid + 1, hi]`.

**Dừng:** `lo <= mid < hi` nên mỗi vòng khoảng giảm ít nhất 1 (thực tế giảm khoảng một nửa). Khi `lo == hi`, khoảng chỉ còn một số và nó phải là `B`.

**Độ phức tạp:** gọi `n` là số phiên bản.

| Cách | Số lần gọi API / Time | Space |
|---|---|---|
| 1. Duyệt tuần tự | `O(n)` (tới `n` lần) | `O(1)` |
| 2. Binary search `lo < hi` | `O(log n)` (≤ `ceil(log2 n)`) | `O(1)` |
| 3. Binary search `lo <= hi` + `ans` | `O(log n)` (≈ `floor(log2 n) + 1`) | `O(1)` |
| 4. `bisect_left(..., key=)` | `O(log n)` | `O(1)` (range không tạo list) |

In [12]:
# Kiểm chứng giới hạn ceil(log2 n) cho khuôn lo < hi trên mọi vị trí bad với vài n
for n in [1, 2, 3, 7, 8, 100, 1000]:
    worst = 0
    for bad in range(1, n + 1):
        set_bad(bad)
        assert first_bad_binary(n) == bad
        worst = max(worst, CALLS)
    print(f"n = {n:>4}: số lần gọi tối đa = {worst}, ceil(log2 n) = {math.ceil(math.log2(n)) if n > 1 else 0}")

n =    1: số lần gọi tối đa = 0, ceil(log2 n) = 0
n =    2: số lần gọi tối đa = 1, ceil(log2 n) = 1
n =    3: số lần gọi tối đa = 2, ceil(log2 n) = 2
n =    7: số lần gọi tối đa = 3, ceil(log2 n) = 3
n =    8: số lần gọi tối đa = 3, ceil(log2 n) = 3
n =  100: số lần gọi tối đa = 7, ceil(log2 n) = 7
n = 1000: số lần gọi tối đa = 10, ceil(log2 n) = 10


## 11. Bộ test

In [13]:
def run(n, bad, f):
    set_bad(bad)
    return f(n)

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS ({actual})")

sol = Solution()
check("mẫu 1: n=5, bad=4", run(5, 4, sol.firstBadVersion), 4)
check("mẫu 2: n=1, bad=1", run(1, 1, sol.firstBadVersion), 1)
check("bad = 1 (tất cả lỗi)", run(100, 1, sol.firstBadVersion), 1)
check("bad = n (chỉ bản cuối lỗi)", run(100, 100, sol.firstBadVersion), 100)
check("n = 2, bad = 2", run(2, 2, sol.firstBadVersion), 2)
check("n lớn nhất, bad = n", run(2**31 - 1, 2**31 - 1, sol.firstBadVersion), 2**31 - 1)
check("n lớn nhất, bad ở giữa", run(2**31 - 1, 1702766719, sol.firstBadVersion), 1702766719)
set_bad(2**31 - 1); sol.firstBadVersion(2**31 - 1)
assert CALLS <= 31
print("n = 2^31 - 1 chỉ cần", CALLS, "lần gọi API")

mẫu 1: n=5, bad=4                  PASS (4)
mẫu 2: n=1, bad=1                  PASS (1)
bad = 1 (tất cả lỗi)               PASS (1)
bad = n (chỉ bản cuối lỗi)         PASS (100)
n = 2, bad = 2                     PASS (2)
n lớn nhất, bad = n                PASS (2147483647)
n lớn nhất, bad ở giữa             PASS (1702766719)
n = 2^31 - 1 chỉ cần 30 lần gọi API


In [14]:
import random
rng = random.Random(278)
funcs = [first_bad_linear, first_bad_binary, first_bad_with_ans, first_bad_bisect, Solution().firstBadVersion]
for _ in range(500):
    n = rng.randint(1, 300)
    bad = rng.randint(1, n)
    assert all(run(n, bad, f) == bad for f in funcs), (n, bad)
for _ in range(500):
    n = rng.randint(1, 2**31 - 1)
    bad = rng.randint(1, n)
    assert all(run(n, bad, f) == bad for f in funcs[1:]), (n, bad)
print("Mọi cách đúng trên 1000 bộ (n, bad) ngẫu nhiên")

Mọi cách đúng trên 1000 bộ (n, bad) ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: `hi = mid - 1` khi `isBadVersion(mid)` đúng (trong khuôn `lo < hi`)

Làm mất `mid` — có thể chính là đáp án.

In [15]:
def wrong_drop_mid(n):
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if isBadVersion(mid):
            hi = mid - 1          # SAI: bỏ mất mid
        else:
            lo = mid + 1
    return lo

print("n=5, bad=3: wrong =", run(5, 3, wrong_drop_mid), "| đúng = 3")

n=5, bad=3: wrong = 2 | đúng = 3


### Lỗi 2: `lo = mid` thay vì `lo = mid + 1` → vòng lặp vô hạn

Khi `hi = lo + 1`, `mid = lo`; nếu `isBadVersion(mid)` sai và ta gán `lo = mid`, khoảng **không đổi** → lặp mãi. Ta mô phỏng có giới hạn bước:

In [16]:
def wrong_lo_mid_limited(n, max_steps=50):
    lo, hi = 1, n
    steps = 0
    while lo < hi:
        steps += 1
        if steps > max_steps:
            return f"kẹt ở lo = {lo}, hi = {hi} sau {max_steps} bước -> vô hạn"
        mid = (lo + hi) // 2
        if isBadVersion(mid):
            hi = mid
        else:
            lo = mid              # SAI
    return lo

print(run(5, 4, wrong_lo_mid_limited))

kẹt ở lo = 3, hi = 4 sau 50 bước -> vô hạn


### Lỗi 3: Trộn khuôn — `while lo <= hi` nhưng lại `hi = mid`

Với `lo == hi` và `isBadVersion(mid)` đúng, `hi = mid` không đổi gì → vô hạn. **Quy tắc:** `lo < hi` đi với `hi = mid`; `lo <= hi` đi với `hi = mid - 1` (và cần biến `ans`).

In [17]:
def wrong_mixed_limited(n, max_steps=50):
    lo, hi = 1, n
    steps = 0
    while lo <= hi:               # SAI khi đi kèm hi = mid
        steps += 1
        if steps > max_steps:
            return f"kẹt ở lo = hi = {lo} -> vô hạn"
        mid = (lo + hi) // 2
        if isBadVersion(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print(run(5, 4, wrong_mixed_limited))

kẹt ở lo = hi = 4 -> vô hạn


### Lỗi 4: Bắt đầu từ `0` hoặc trả `hi + 1`/`mid`

Phiên bản đánh số từ **1**. Trả `mid` của vòng cuối là sai vì `mid` cuối có thể là bản tốt. Luôn trả `lo` (khuôn `lo < hi`) hoặc `ans` (khuôn `lo <= hi`).

In [18]:
def wrong_return_mid(n):
    lo, hi = 1, n
    mid = lo
    while lo < hi:
        mid = (lo + hi) // 2
        if isBadVersion(mid):
            hi = mid
        else:
            lo = mid + 1
    return mid                    # SAI

print("n=10, bad=7: wrong =", run(10, 7, wrong_return_mid), "| đúng = 7")

n=10, bad=7: wrong = 6 | đúng = 7


### Lỗi 5: Gọi API thừa — ví dụ gọi `isBadVersion(mid)` hai lần trong một vòng

Đề yêu cầu **tối thiểu số lần gọi**. Lưu kết quả vào biến nếu cần dùng lại.

## 13. Mẹo nhận dạng pattern

- Có một **điều kiện đơn điệu** theo chỉ số (`F...F T...T`) và cần **điểm chuyển** → binary search "first true" (khuôn `lo < hi`, `hi = mid`, `lo = mid + 1`).
- Đề nói "**gọi API ít nhất**", "**`n` tới 2^31**" → cần `O(log n)`.
- Cụm từ "tất cả sau đó đều …", "từ một thời điểm trở đi …" → tính đơn điệu.

Bài liên quan: 35 Search Insert Position, 374 Guess Number Higher or Lower, 69 Sqrt(x), 875 Koko Eating Bananas (binary search trên đáp án), 34 Find First and Last Position.

## 14. Biến thể

### 14.1. Khuôn tổng quát "first true" trên một vị từ bất kỳ

In [19]:
def first_true(lo: int, hi: int, pred) -> int:
    # Giả sử pred đơn điệu F..F T..T trên [lo, hi] và pred(hi) = True
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if pred(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

print("căn bậc hai nguyên của 2026 (số x nhỏ nhất có (x+1)^2 > 2026):", first_true(0, 2026, lambda x: (x + 1) ** 2 > 2026))
print("số nhỏ nhất có 2^x >= 10^9:", first_true(0, 64, lambda x: 2 ** x >= 10**9))

căn bậc hai nguyên của 2026 (số x nhỏ nhất có (x+1)^2 > 2026): 45
số nhỏ nhất có 2^x >= 10^9: 30


### 14.2. "Last false" — phiên bản tốt cuối cùng

Chính là `first_true - 1`. Hoặc viết khuôn đối xứng với `mid` làm tròn **lên** để tránh kẹt.

In [20]:
def last_good(n):
    lo, hi = 0, n                 # 0 nghĩa là "không có bản tốt"
    while lo < hi:
        mid = (lo + hi + 1) // 2  # làm tròn lên!
        if isBadVersion(mid):
            hi = mid - 1
        else:
            lo = mid
    return lo

for n, bad in [(10, 7), (5, 1), (5, 5)]:
    print(f"n = {n}, bad = {bad}: bản tốt cuối = {run(n, bad, last_good)}")

n = 10, bad = 7: bản tốt cuối = 6
n = 5, bad = 1: bản tốt cuối = 0
n = 5, bad = 5: bản tốt cuối = 4


### 14.3. Không biết `n` (dãy vô hạn) — exponential search

Hỏi `1, 2, 4, 8, ...` tới khi gặp bản lỗi, rồi binary search trong `[hi/2 + 1, hi]`. Vẫn `O(log B)` lần gọi với `B` là đáp án.

In [21]:
def first_bad_unbounded() -> int:
    hi = 1
    while not isBadVersion(hi):
        hi *= 2
    return first_true(hi // 2 + 1 if hi > 1 else 1, hi, isBadVersion)

for bad in [1, 2, 37, 10**12]:
    set_bad(bad)
    print(f"bad = {bad}: tìm được {first_bad_unbounded()} sau {CALLS} lần gọi")

bad = 1: tìm được 1 sau 1 lần gọi
bad = 2: tìm được 2 sau 2 lần gọi
bad = 37: tìm được 37 sau 12 lần gọi
bad = 1000000000000: tìm được 1000000000000 sau 80 lần gọi


## 15. Bài tập (tự làm — cell chỉ định nghĩa, không gọi)

**Bài 1 — Trace tay:** `n = 16, bad = 11`. Liệt kê các `mid` mà khuôn `lo < hi` sẽ hỏi.

In [22]:
# TODO
predict_mids = None
# assert predict_mids == <xem đáp án>

**Bài 2 — Điền chỗ trống** khuôn `lo < hi`.

In [23]:
def exercise_first_bad(n: int) -> int:
    lo, hi = 1, n
    while lo < hi:
        mid = lo + (hi - lo) // 2
        # TODO: nếu mid lỗi thì ..., ngược lại ...
        pass
    return lo

# assert run(10, 7, exercise_first_bad) == 7

**Bài 3 — Guess Number Higher or Lower (374):** API `guess(x)` trả `-1` nếu số bí mật **nhỏ hơn** `x`, `1` nếu **lớn hơn**, `0` nếu đúng. Viết `exercise_guess(n, guess)`.

In [24]:
def exercise_guess(n: int, guess) -> int:
    # TODO
    pass

# secret = 6; g = lambda x: (secret > x) - (secret < x)
# assert exercise_guess(10, g) == 6

**Bài 4 — Search Insert Position (35):** mảng tăng, trả vị trí đầu tiên `i` có `nums[i] >= target` (hoặc `len(nums)`).

In [25]:
def exercise_search_insert(nums: list[int], target: int) -> int:
    # TODO
    pass

# assert exercise_search_insert([1, 3, 5, 6], 5) == 2 and exercise_search_insert([1, 3, 5, 6], 7) == 4

**Bài 5 — Sqrt(x) (69):** phần nguyên của căn bậc hai, chỉ dùng phép nhân và binary search.

In [26]:
def exercise_isqrt(x: int) -> int:
    # TODO
    pass

# assert [exercise_isqrt(v) for v in [0, 1, 4, 8, 2**31 - 1]] == [0, 1, 2, 2, 46340]

**Bài 6 — Mini challenge — Find First and Last Position (34):** mảng `nums` tăng dần (có thể trùng), trả `[đầu, cuối]` vị trí của `target`, hoặc `[-1, -1]`. Yêu cầu `O(log n)`: dùng **hai lần** khuôn first-true.

In [27]:
def exercise_search_range(nums: list[int], target: int) -> list[int]:
    # TODO: first = vị trí đầu tiên nums[i] >= target; last = (vị trí đầu tiên nums[i] > target) - 1
    pass

# assert exercise_search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4] and exercise_search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `lo = 1, hi = 16` | `mid = 8` → F | mid tiếp theo trong `[9, 16]` là 12 |
| 2 | mục 5 đoạn 4 | lỗi → giữ mid | `hi = mid` / `lo = mid + 1` |
| 3 | `guess(mid) == 0` → trả | `-1` → số nhỏ hơn → `hi = mid - 1` | khuôn `lo <= hi` |
| 4 | vị từ `nums[i] >= target` | `hi = len(nums)` (có thể chèn cuối) | khuôn first-true |
| 5 | vị từ `m * m > x` | first-true trên `[0, x + 1]` | trả `first - 1` |
| 6 | hai vị từ: `>= target` và `> target` | viết hàm `first_true` dùng chung | nếu `first == len` hoặc `nums[first] != target` → `[-1, -1]` |

## 17. Đáp án tham khảo

In [28]:
answer_mids = [8, 12, 10, 11]
set_bad(11)
lo, hi, mids = 1, 16, []
while lo < hi:
    mid = lo + (hi - lo) // 2
    mids.append(mid)
    if isBadVersion(mid):
        hi = mid
    else:
        lo = mid + 1
assert mids == answer_mids
print("Bài 1:", answer_mids)

def answer_first_bad(n):
    lo, hi = 1, n
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if isBadVersion(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_guess(n, guess):
    lo, hi = 1, n
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        g = guess(mid)
        if g == 0:
            return mid
        if g < 0:
            hi = mid - 1
        else:
            lo = mid + 1
    return -1

def answer_search_insert(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_isqrt(x):
    lo, hi = 0, x + 1             # first m với m*m > x
    while lo < hi:
        mid = (lo + hi) // 2
        if mid * mid > x:
            hi = mid
        else:
            lo = mid + 1
    return lo - 1

def answer_search_range(nums, target):
    def first(pred):
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if pred(nums[mid]):
                hi = mid
            else:
                lo = mid + 1
        return lo
    a = first(lambda v: v >= target)
    if a == len(nums) or nums[a] != target:
        return [-1, -1]
    b_ = first(lambda v: v > target) - 1
    return [a, b_]

assert run(10, 7, answer_first_bad) == 7
secret = 6
assert answer_guess(10, lambda x: (secret > x) - (secret < x)) == 6
assert answer_search_insert([1, 3, 5, 6], 5) == 2 and answer_search_insert([1, 3, 5, 6], 7) == 4
assert [answer_isqrt(v) for v in [0, 1, 4, 8, 2**31 - 1]] == [0, 1, 2, 2, 46340]
assert answer_search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4] and answer_search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert answer_search_range([], 0) == [-1, -1] and answer_search_range([2, 2, 2], 2) == [0, 2]
print("Tất cả đáp án PASS")

Bài 1: [8, 12, 10, 11]
Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def firstBadVersion(self, n: int) -> int:
        lo, hi = 1, n                    # đáp án luôn trong [lo, hi]
        while lo < hi:
            mid = lo + (hi - lo) // 2    # làm tròn xuống
            if isBadVersion(mid):
                hi = mid                 # giữ mid
            else:
                lo = mid + 1             # bỏ mid
        return lo
```

**Câu thần chú:** *"F...F T...T → first true: đúng thì `hi = mid`, sai thì `lo = mid + 1`, `while lo < hi`, trả `lo`."*

**Checklist:**
- [ ] `lo = 1` (phiên bản đánh số từ 1), `hi = n`.
- [ ] `while lo < hi` đi cùng `hi = mid` — không trộn với `lo <= hi`.
- [ ] `lo = mid + 1` (không phải `lo = mid`) để tránh lặp vô hạn.
- [ ] Mỗi vòng gọi API **một lần**.
- [ ] Trả `lo`, không trả `mid`.